In [45]:
import pandas as pd
import re
import os

In [46]:
df = pd.read_excel("data/Chair-Cochair.xlsx", sheet_name="Chair-cochairs", skiprows=2)

In [47]:
df.head(3)

,Num,Symposia,Role,Thai,Name,Affiliation,Email,Ignore
0,1,"Semiconductors, Photonic Materials and Devices",Chair,รองศาสตราจารย์ ดร. อนุชา วัชระภาสร,Assoc. Prof. Dr. Anucha Watcharapasorn,Chiang Mai University,anucha.w@cmu.ac.th,NaN
1,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ศาสตราจารย์ ดร. วิษณุ เพชรภา,Prof. Dr. Wisanu Pecharapa,King Mongkut's Institute of Technology Ladkrabang,wisanu.pe@kmitl.ac.th,NaN
2,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ดร. ทศพร เลิศวณิชผล,Dr. Tossaporn Lertvanithphol,National Electronics and Computer Technology C...,tossaporn.ler@nectec.or.th,NaN


In [48]:
# Remove rows where "Ignore" is "YES"
filt = df["Ignore"] == "YES"
display(df[filt])
df = df[~filt]

,Num,Symposia,Role,Thai,Name,Affiliation,Email,Ignore
10,2,"Polymers, Biomaterials and Medical Devices",Co-chair,ดร. ศรภัทร นิยมสินธุ์,Dr. Sorapat Niyomsin,Chulalongkorn University,sorapat.n@chula.ac.th,YES


In [49]:
def extract_name(comb_txt):
    pattern = re.compile(
        r"^(?:Assoc\. Prof\. Dr\.|Asst\. Prof\. Dr\.|Prof\. Dr\.|Dr\.)\s+(.*)$"
    )
    m = pattern.match(comb_txt.strip())
    if m:
        return m.group(1)
    return ""


df["name_strip"] = df["Name"].apply(extract_name)
df.head(3)

,Num,Symposia,Role,Thai,Name,Affiliation,Email,Ignore,name_strip
0,1,"Semiconductors, Photonic Materials and Devices",Chair,รองศาสตราจารย์ ดร. อนุชา วัชระภาสร,Assoc. Prof. Dr. Anucha Watcharapasorn,Chiang Mai University,anucha.w@cmu.ac.th,NaN,Anucha Watcharapasorn
1,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ศาสตราจารย์ ดร. วิษณุ เพชรภา,Prof. Dr. Wisanu Pecharapa,King Mongkut's Institute of Technology Ladkrabang,wisanu.pe@kmitl.ac.th,NaN,Wisanu Pecharapa
2,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ดร. ทศพร เลิศวณิชผล,Dr. Tossaporn Lertvanithphol,National Electronics and Computer Technology C...,tossaporn.ler@nectec.or.th,NaN,Tossaporn Lertvanithphol


In [50]:
def get_title(row):
    name = row["Name"]
    name_strip = row["name_strip"]
    title = name.replace(name_strip, "").strip()
    return title.strip()


df["title"] = df.apply(get_title, axis=1)
df.head(3)

,Num,Symposia,Role,Thai,Name,Affiliation,Email,Ignore,name_strip,title
0,1,"Semiconductors, Photonic Materials and Devices",Chair,รองศาสตราจารย์ ดร. อนุชา วัชระภาสร,Assoc. Prof. Dr. Anucha Watcharapasorn,Chiang Mai University,anucha.w@cmu.ac.th,NaN,Anucha Watcharapasorn,Assoc. Prof. Dr.
1,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ศาสตราจารย์ ดร. วิษณุ เพชรภา,Prof. Dr. Wisanu Pecharapa,King Mongkut's Institute of Technology Ladkrabang,wisanu.pe@kmitl.ac.th,NaN,Wisanu Pecharapa,Prof. Dr.
2,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ดร. ทศพร เลิศวณิชผล,Dr. Tossaporn Lertvanithphol,National Electronics and Computer Technology C...,tossaporn.ler@nectec.or.th,NaN,Tossaporn Lertvanithphol,Dr.


In [51]:
def create_filename(row):
    X = row["Num"]
    Role = row["Role"]
    Name = row["name_strip"]
    return f"Invitation IUMRS-ICEM2026_Symp{X}_{Role}_{Name}"


df["filename"] = df.apply(create_filename, axis=1)
df["filename"].values

<StringArray>
[         'Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha Watcharapasorn',
            'Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisanu Pecharapa',
    'Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossaporn Lertvanithphol',
          'Invitation IUMRS-ICEM2026_Symp1_Co-chair_Doldet Tantraviwat',
       'Invitation IUMRS-ICEM2026_Symp1_Co-chair_Thutiyaporn Thiwawong',
                'Invitation IUMRS-ICEM2026_Symp2_Chair_Winita Punyodom',
            'Invitation IUMRS-ICEM2026_Symp2_Co-chair_Kannika Sahakaro',
                'Invitation IUMRS-ICEM2026_Symp2_Co-chair_Sukunya Ross',
           'Invitation IUMRS-ICEM2026_Symp2_Co-chair_Panya Sunintaboon',
    'Invitation IUMRS-ICEM2026_Symp2_Co-chair_Patchara Punyamoonwongsa',
            'Invitation IUMRS-ICEM2026_Symp3_Chair_Adisorn Tuantranont',
             'Invitation IUMRS-ICEM2026_Symp3_Co-chair_Chanpen Karuwan',
          'Invitation IUMRS-ICEM2026_Symp3_Co-chair_Kamonwad Ngamchuea',
            'Invitation IUMRS-ICEM202

In [52]:
df["docx"] = df["filename"] + ".docx"
df["pdf"] = df["filename"] + ".pdf"
df["html"] = df["filename"] + ".html"

filt_chair = df["Role"] == "Chair"
filt_cochair = df["Role"] == "Co-chair"

df.loc[filt_chair, "docx_path"] = "output_docx/chair"
df.loc[filt_chair, "pdf_path"] = "output_pdf/chair"
df.loc[filt_chair, "html_path"] = "output_html/chair"

df.loc[filt_cochair, "docx_path"] = "output_docx/cochair"
df.loc[filt_cochair, "pdf_path"] = "output_pdf/cochair"
df.loc[filt_cochair, "html_path"] = "output_html/cochair"

df.loc[filt_chair, "template_docx"] = "templates/Invitation IUMRS-ICEM2026_Chair_v2_template.docx"
df.loc[filt_chair, "template_html"] = "templates/Invitation IUMRS-ICEM2026_Chair_v2_template.html"

df.loc[filt_cochair, "template_docx"] = "templates/Invitation IUMRS-ICEM2026_Co-chair_v2_template.docx"
df.loc[filt_cochair, "template_html"] = "templates/Invitation IUMRS-ICEM2026_Co-chair_v2_template.html"

df.head(3)

,Num,Symposia,Role,Thai,Name,Affiliation,Email,Ignore,name_strip,title,filename,docx,pdf,html,docx_path,pdf_path,html_path,template_docx,template_html
0,1,"Semiconductors, Photonic Materials and Devices",Chair,รองศาสตราจารย์ ดร. อนุชา วัชระภาสร,Assoc. Prof. Dr. Anucha Watcharapasorn,Chiang Mai University,anucha.w@cmu.ac.th,NaN,Anucha Watcharapasorn,Assoc. Prof. Dr.,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...,output_docx/chair,output_pdf/chair,output_html/chair,templates/Invitation IUMRS-ICEM2026_Chair_v2_t...,templates/Invitation IUMRS-ICEM2026_Chair_v2_t...
1,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ศาสตราจารย์ ดร. วิษณุ เพชรภา,Prof. Dr. Wisanu Pecharapa,King Mongkut's Institute of Technology Ladkrabang,wisanu.pe@kmitl.ac.th,NaN,Wisanu Pecharapa,Prof. Dr.,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...,output_docx/cochair,output_pdf/cochair,output_html/cochair,templates/Invitation IUMRS-ICEM2026_Co-chair_v...,templates/Invitation IUMRS-ICEM2026_Co-chair_v...
2,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ดร. ทศพร เลิศวณิชผล,Dr. Tossaporn Lertvanithphol,National Electronics and Computer Technology C...,tossaporn.ler@nectec.or.th,NaN,Tossaporn Lertvanithphol,Dr.,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...,output_docx/cochair,output_pdf/cochair,output_html/cochair,templates/Invitation IUMRS-ICEM2026_Co-chair_v...,templates/Invitation IUMRS-ICEM2026_Co-chair_v...


In [53]:
# Create folder if not exist
for path in df["docx_path"].unique():
    os.makedirs(path, exist_ok=True)
for path in df["pdf_path"].unique():
    os.makedirs(path, exist_ok=True)
for path in df["html_path"].unique():
    os.makedirs(path, exist_ok=True)

In [54]:
# For prototyping, let's just work with the first group (Num=1)

# dfG = df.groupby(by="Num")
# print(dfG.groups.keys())
# _df = dfG.get_group(1).copy()

# filt_chair = _df["Role"] == "Chair"
# filt_cochair = _df["Role"] == "Co-chair"

# # Assuming there's only one chair per group, get the chair's email
# chair_email = _df[filt_chair]["Email"].values[0]

# # Mandatory emails to cc for co-chairs
# anucha_email = "anucha@stanfordalumni.org"

# # For chairs, cc to Anucha. For co-chairs, cc to both chair and Anucha.
# _df.loc[filt_chair, "cc_email"] = anucha_email
# _df.loc[filt_cochair, "cc_email"] = ",".join([chair_email, anucha_email])

In [55]:

def cc_email(_df):
    filt_chair = _df["Role"] == "Chair"
    filt_cochair = _df["Role"] == "Co-chair"

    # Assuming there's only one chair per group, get the chair's email
    chair_email = _df[filt_chair]["Email"].values[0]

    # Mandatory emails to cc for co-chairs
    anucha_email = "anucha@stanfordalumni.org"

    # For chairs, cc to Anucha. For co-chairs, cc to both chair and Anucha.
    _df.loc[filt_chair, "cc_email"] = anucha_email
    _df.loc[filt_cochair, "cc_email"] = ",".join([chair_email, anucha_email])

    return _df


df = df.groupby(by="Num").apply(cc_email).reset_index().drop(columns="level_1")
df.head(3)


,Num,Symposia,Role,Thai,Name,Affiliation,Email,Ignore,name_strip,title,filename,docx,pdf,html,docx_path,pdf_path,html_path,template_docx,template_html,cc_email
0,1,"Semiconductors, Photonic Materials and Devices",Chair,รองศาสตราจารย์ ดร. อนุชา วัชระภาสร,Assoc. Prof. Dr. Anucha Watcharapasorn,Chiang Mai University,anucha.w@cmu.ac.th,NaN,Anucha Watcharapasorn,Assoc. Prof. Dr.,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...,output_docx/chair,output_pdf/chair,output_html/chair,templates/Invitation IUMRS-ICEM2026_Chair_v2_t...,templates/Invitation IUMRS-ICEM2026_Chair_v2_t...,anucha@stanfordalumni.org
1,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ศาสตราจารย์ ดร. วิษณุ เพชรภา,Prof. Dr. Wisanu Pecharapa,King Mongkut's Institute of Technology Ladkrabang,wisanu.pe@kmitl.ac.th,NaN,Wisanu Pecharapa,Prof. Dr.,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...,output_docx/cochair,output_pdf/cochair,output_html/cochair,templates/Invitation IUMRS-ICEM2026_Co-chair_v...,templates/Invitation IUMRS-ICEM2026_Co-chair_v...,"anucha.w@cmu.ac.th,anucha@stanfordalumni.org"
2,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ดร. ทศพร เลิศวณิชผล,Dr. Tossaporn Lertvanithphol,National Electronics and Computer Technology C...,tossaporn.ler@nectec.or.th,NaN,Tossaporn Lertvanithphol,Dr.,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...,output_docx/cochair,output_pdf/cochair,output_html/cochair,templates/Invitation IUMRS-ICEM2026_Co-chair_v...,templates/Invitation IUMRS-ICEM2026_Co-chair_v...,"anucha.w@cmu.ac.th,anucha@stanfordalumni.org"


In [56]:
# Email header
def gen_email_subject(row):
    role = row["Role"]
    return f"Symposium {row['Num']} : Invitation as a {role} at IUMRS-ICEM2026 ({row['Name']})"


df["email_subject"] = df.apply(gen_email_subject, axis=1)

df["email_subject"].values

<StringArray>
[         'Symposium 1 : Invitation as a Chair at IUMRS-ICEM2026 (Assoc. Prof. Dr. Anucha Watcharapasorn)',
                   'Symposium 1 : Invitation as a Co-chair at IUMRS-ICEM2026 (Prof. Dr. Wisanu Pecharapa)',
                 'Symposium 1 : Invitation as a Co-chair at IUMRS-ICEM2026 (Dr. Tossaporn Lertvanithphol)',
          'Symposium 1 : Invitation as a Co-chair at IUMRS-ICEM2026 (Assoc. Prof. Dr. Doldet Tantraviwat)',
        'Symposium 1 : Invitation as a Co-chair at IUMRS-ICEM2026 (Asst. Prof. Dr. Thutiyaporn Thiwawong)',
                'Symposium 2 : Invitation as a Chair at IUMRS-ICEM2026 (Assoc. Prof. Dr. Winita Punyodom)',
            'Symposium 2 : Invitation as a Co-chair at IUMRS-ICEM2026 (Assoc. Prof. Dr. Kannika Sahakaro)',
                'Symposium 2 : Invitation as a Co-chair at IUMRS-ICEM2026 (Assoc. Prof. Dr. Sukunya Ross)',
           'Symposium 2 : Invitation as a Co-chair at IUMRS-ICEM2026 (Assoc. Prof. Dr. Panya Sunintaboon)',
     'Symposiu

In [57]:
df.columns

Index(['Num', 'Symposia', 'Role', 'Thai', 'Name', 'Affiliation', 'Email',
       'Ignore', 'name_strip', 'title', 'filename', 'docx', 'pdf', 'html',
       'docx_path', 'pdf_path', 'html_path', 'template_docx', 'template_html',
       'cc_email', 'email_subject'],
      dtype='str')

In [58]:
df.to_excel("output_excel/S01_data.xlsx", index=False)